# Dots3-Note on ARC-AGI-2: a 4×L4 submission

This notebook writes `submission.json` for the hidden ARC-AGI-2 test set. The whole run (model load, solving, write-out)
must fit in Kaggle's 12-hour limit on 4×NVIDIA L4 (24 GB each), with no internet access.

**Approach**
- **Model.** *Dots3-Note lb144r*, attached as a Kaggle Model (4 variations of ~15 GB): Dots3-Note with 144 experts per MoE layer, and GPTQ
  3/2-bit expert weights in `lowbit-L*.pt`. A small fine-tuned weight delta (`p1s_delta.safetensors`) is overlaid at
  load time.
- **Serving.** A patched vLLM with tensor parallelism 4, MTP-3 speculative decoding, an FP8 KV cache and contexts up to
  262k tokens. Long sessions keep most of their KV cache in host memory.
- **Solver.** `solve.py` keeps 64 requests in flight. Each test input gets two attempts. Token caps shrink as the
  deadline nears, and an end-game phase forces every open attempt to conclude in time.

**Levers** (all set in the configuration cell, and all checked against the runtime's mode file)

| lever | what | why |
|---|---|---|
| storage | the checkpoint, env and CUDA tars are copied to local disk by a parallel copier with retries and hedged reads | the input mounts can be slow (10–100 MB/s), and a single reader stalls |
| startup guard | every deadline counts from session start; the server gets one restart before it is ready; hard stop at +42,600 s | a slow or failed start never costs the submission |
| memory | KV pool 3 GiB/GPU, compact GPU row cache (host row format kept on the GPU), KV-b rebuild, lazy penalties, compact parameters | the 4×24 GB budget fits the model, CUDA graphs and 64 long sequences |
| throughput | host all-reduce at 6 bits (main path 4f/5 + error feedback), L4-tuned FP8 GEMM tiles, offload attention v2 | the L4s talk only over PCIe through the host, so communication bytes bound decode speed |
| startup cost | pinned kernel choices, a shipped Triton cache, 2 s readiness polling | minutes of autotuning and JIT become seconds |

**Notebook layout:** configuration → helpers → stage 1 inputs → stage 2 run mode → stage 3 runtime and environment
→ stage 4 model server → stage 5 solver → stage 6 submission check.

The runtime itself (launcher `run_l4.py`, solver `solve.py`, vLLM plugin and kernels, and the Python 3.12 environment) ships in the private dataset
*dots3-note-arc-runtime*. This notebook runs that launcher stage by stage and shows each stage's state.

## Configuration

In [ ]:
import hashlib, json, os, re, subprocess, sys, time
from pathlib import Path

# -- inputs and outputs (Kaggle mounts every attached source under /kaggle/input)
INPUT_DIR = os.environ.get("ARC_INPUT", "/kaggle/input")
WORK_DIR = Path(os.environ.get("ARC_WORK", "/kaggle/working"))
RUNTIME_DATASET = "dots3-note-arc-runtime"          # code bundle: run_l4.py, solve.py, plugin, kernels, mode files
MODEL_OWNER, MODEL_SLUG = "notnitsuj", "dots3-note-lb144r"   # Kaggle Model, mounted at .../<model>/pytorch/<variation>/<version>/
MODEL_PARTS = ["lb144r-p1", "lb144r-p2", "lb144r-p3", "lb144r-p4"]   # the checkpoint split into 4 variations (~15 GB each)
CKPT_MERGE_DIR = Path(os.environ.get("ARC_CKPT_MERGE_DIR", "/tmp/arc_ckpt_parts"))   # links to the parts' files
CKPT_MARKER = "lowbit-L45.pt"                   # last expert file: marks a checkpoint directory
MODE_FILE = "mode_submit.json"                      # the runtime's submission mode (pinned below)

# -- deadlines, seconds. S0 = session start (container start); every guard time counts from it
DEADLINE_TOTAL_S = 42150       # solver budget from the launcher start ...
STARTUP_GUARD = {'anchor': 'session', 'anchor_max_age_s': 3600, 'deadline_abs_s': 42150, 'kill_by_s': 42600, 'retries': 1, 'retry_min_left_s': 1800, 'hang_s': 2400, 'solve_min_s': 120, 'find_index': True}
# ... capped at S0 + deadline_abs_s; at S0 + kill_by_s every child is killed, submission.json repaired, exit 0
BACKSTOP_GRACE_S = 180                              # notebook-side backstop, only if the launcher outlives its hard stop

# -- save run (interactive "Save Version"): a short smoke test on the public evaluation split, then the fallback echo
SAVE_RUN = {"challenges": "evaluation", "score": True, "deadline_total_s": 1200,
            "note": "Save run: short smoke test on the public evaluation split."}
SAVE_WATCHDOG_S, SAVE_WATCHDOG_KILL_S = 1800, 90    # TERM the save run after 1800 s, KILL 90 s later

# -- model server
# vLLM serve arguments (tensor parallel 4; the runtime adds the port, TP size and memory utilisation)
SERVE_ARGS = [
    "--language-model-only",
    "--trust-remote-code",
    "--disable-custom-all-reduce",
    "--kv-cache-dtype", "fp8",
    "--max-num-batched-tokens", "1024",
    "--max-num-seqs", "64",
    "--max-model-len", "262144",
    "--block-size", "32",
    "--enable-prefix-caching",
    "--kernel-config", "{\"enable_flashinfer_autotune\": false}",
    "--attention-config", "{\"flash_attn_version\": 2}",
    "--compilation-config", "{\"cudagraph_mode\": \"PIECEWISE\", \"cudagraph_capture_sizes\": [8, 16, 32, 64, 96, 128, 160, 192, 224, 240, 256], \"pass_config\": {\"fuse_allreduce_rms\": false}}",
    "--speculative-config", "{\"method\":\"mtp\",\"num_speculative_tokens\":3}",
    "--prefix-cache-retention-interval", "1280",
]
KV_CACHE_GIB = 3.0                                  # explicit KV pool per GPU
GPU_MEMORY_UTILIZATION = 0.905
DELTA_FILE = "p1s_delta.safetensors"                     # fine-tuned delta, overlaid at load time
TRITON_CACHE = "triton_cache_lv5.tarbin"               # precompiled Triton kernels for L4
READY_POLL_S = 2

# -- solver
# Solver (solve.py): deadline-aware 2-attempt sampling against the local vLLM server
SOLVER_ARGS = [
    "--concurrency", "64",             # requests in flight (= --max-num-seqs)
    "--cap1", "240000",                # token cap, attempt 1
    "--cap2", "120000",                # token cap, attempt 2
    "--min-cap", "24000",              # smallest attempt-1 cap the deadline planner may shrink to
    "--min-cap2", "40000",             # smallest attempt-2 cap
    "--endgame-s", "450",              # last seconds: no new attempts, force conclusions
    "--attempts", "2",                 # attempts per test input (pass@2)
    "--kv-admit", "0.85",              # admit new work below this KV-cache use
    "--session-chunk", "2048",         # continuation chunk size in a session (tokens)
    "--budget-margin-s", "300",        # the deadline planner sizes the remaining work to end this long before the deadline
]

# -- page-cache prefetch and local staging of the checkpoint / env / CUDA tars
PREFETCH = {'threads': 16, 'block_mb': 8, 'max_gib': 90, 'stage': 'always', 'stage_dir': '/tmp/arc_ckpt', 'stage_threads': 32, 'seg_mb': 64, 'robust': True, 'stall_s': 120, 'max_attempts': 8, 'stage_env': True, 'barrier_seg_mb': 16, 'barrier_stall_s': 30}

# -- startup kernel choices pinned to what the gates picked on every Kaggle 4xL4 run (no gate time at startup)
GATE_PINS = {'ARC_OFFLOAD_HT_EPOCH': '1', 'ARC_OFFLOAD_FILL_SPLIT': '1', 'ARC_OFFLOAD_FILL_B': '32', 'ARC_OFFLOAD_FILL_WARPS': '4', 'ARC_OFFLOAD_FILL_CTAS': '8', 'ARC_OFFLOAD_TWO_CHOICE': '1', 'ARC_SWA_FUSED': '2', 'ARC_SWA_CUDA': '31', 'ARC_FP8_W8A8': '1', 'ARC_LOWBIT_FUSE': '1', 'ARC_OFFLOAD_PACKED': '0', 'ARC_OFFLOAD_CUDA': '1', 'ARC_OFFLOAD_CUDA_SPLIT': '1', 'ARC_OFFLOAD_CUDA_STAGES': '2', 'ARC_LMHEAD_FP8': '1', 'ARC_SHARD_V2': 'grp,f8=1,BN=128,KB=1,w=4', 'ARC_FAST_TOPK': '1', 'ARC_FAST_TOPK_NT': '512', 'ARC_FAST_TOPK_AGG': '0', 'ARC_FAST_TOPK_MNT': '512', 'ARC_GATE_MERGE': '1'}
HOST_ALLREDUCE_GATE = True       # the one gate still measured: host vs NCCL all-reduce

W8A8_TILE_TABLE = "<in mode file>"
# Environment flags of the vLLM server, grouped by purpose (values = the runtime's mode_submit.json)
ENV_FLAGS = {
    # -- model: Dots3-Note DSA attention on sm89 + the low-bit (3/2-bit) MoE experts
    "ARC_DOTS3_PORTABLE_DSA": "1",
    "ARC_LOWBIT_PAIRS": "1",
    "ARC_LOWBIT_COMPACT": "1",
    "ARC_SWA_FP8": "1",
    "ARC_ROPE_TRIM": "1",
    # -- offload: long-context KV offload to host memory: row caches, formats, fill / spill
    "ARC_DOTS3_OFFLOAD": "1",
    "ARC_OFFLOAD_CACHE_ROWS": "174760",
    "ARC_OFFLOAD_CACHE_ROWS_L": "0:202000,1:362600,5:362600,9:362600,13:362600,17:362600,21:362600,25:362600,29:362600,33:362600,37:362600,41:362600,45:362600",
    "ARC_OFFLOAD_WAYS": "8",
    "ARC_OFFLOAD_RESOLVE": "1",
    "ARC_OFFLOAD_SHARE_HT": "1",
    "ARC_OFFLOAD_MIRROR": "2",
    "ARC_OFFLOAD_OVERLAP": "1",
    "ARC_OFFLOAD_SPILL": "98304",
    "ARC_OFFLOAD_ROW_FMT": "e2m2",
    "ARC_OFFLOAD_WIN_LAZY": "1",
    "ARC_OFFLOAD_ATTN2": "1",
    "ARC_OFFLOAD_ATTN2_KERN": "5",
    "ARC_OFFLOAD_AR_GATE": "1",
    "ARC_OFFLOAD_PF_COMPACT": "32768",
    "ARC_OFFLOAD_CCACHE": "1",
    "ARC_OFFLOAD_PF_OUT": "1",
    "ARC_EMBED_HOST": "1",
    # -- indexer: sparse-attention indexer: sharded over the 4 GPUs, shared between layers
    "ARC_INDEXER_PREFILL_MULT": "4",
    "VLLM_SPARSE_INDEXER_MAX_LOGITS_MB": "128",
    "ARC_SHARD_INDEXER": "1",
    "ARC_SHARD_BS": "128",
    "ARC_SHARD_KLOC": "768",
    "ARC_INDEX_SHARE": "FFSFSFSFSFFSF",
    "ARC_INDEX_SHARE_FREE": "1",
    # -- communication: host-staged all-reduce / all-gather (the L4s have no peer-to-peer link)
    "VLLM_FLASHINFER_ALLREDUCE_BACKEND": "mnnvl",
    "ARC_AG_U16": "1",
    "ARC_HOST_AR_TILE": "4096",
    "ARC_HOST_AR_WARPS": "16",
    "ARC_HOST_AR_BITS": "6",
    "ARC_AR_LB": "1",
    "ARC_AR_LB_MAIN": "4f/5",
    "ARC_AR_LB_EF": "1",
    "ARC_AR_LB_HAD": "8",
    # -- kernels: dense FP8 GEMM tiles, sampling without a device sync
    "ARC_W8A8_V2": "1",
    "ARC_W8A8_V2_CONFIG": W8A8_TILE_TABLE,   # per-(N, K, M) FP8 GEMM tiles measured on L4 (checked by hash)
    "ARC_SHARD_SAMPLE_NOSYNC": "1",
    # -- memory: GPU memory reclaim: allocator, kv_b rebuild, lazy penalties, MLA workspace, compact params
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "ARC_ALLOC_TRIM": "1",
    "ARC_KVB_REBUILD": "1",
    "ARC_PENALTY_LAZY": "1",
    "ARC_MLA_WS_TOKENS": "16384",
    "ARC_PARAM_COMPACT": "1",
    # -- solver sessions: one engine session per attempt (prefix reuse across continuation chunks)
    "ARC_SESSION_EP": "1",
}

# -- identity of the whole mode file (everything above plus the remaining switches; "note" excluded)
MODE_SHA256 = "f97ba865ff8ed6bf1032947e2518ac0fcb56700039c875eef6988470af7d6f8d"
W8A8_TILE_TABLE_SHA256 = "7117f9f73f12e23493dade368eaa071ea1e8792648394c88ae91ddd4f2366764"
LOWBIT_TILE_TABLE_SHA256 = "73a2b00c50d3099f8e87ac861c08add037202c34d0d74200c75cf12c9026fb52"
DRY_RUN = os.environ.get("NB_DRY_RUN") == "1"       # tests only: resolve everything, launch nothing

# -- the checkpoint's files and their sizes in bytes (checked before anything starts, whichever way it is attached)
CKPT_FILES = {
    "LICENSE": 11358,
    "README.md": 12397,
    "README_CN.md": 12090,
    "added_tokens.json": 2795,
    "chat_template.jinja": 6335,
    "config.json": 5001,
    "generation_config.json": 39,
    "lowbit-L01.pt": 1212384361,
    "lowbit-L02.pt": 1212384361,
    "lowbit-L03.pt": 1212384361,
    "lowbit-L04.pt": 1212384361,
    "lowbit-L05.pt": 1212384361,
    "lowbit-L06.pt": 1212384361,
    "lowbit-L07.pt": 1212384361,
    "lowbit-L08.pt": 1212384361,
    "lowbit-L09.pt": 1212384361,
    "lowbit-L10.pt": 1212384361,
    "lowbit-L11.pt": 1212384361,
    "lowbit-L12.pt": 1212384361,
    "lowbit-L13.pt": 1212384361,
    "lowbit-L14.pt": 1212384361,
    "lowbit-L15.pt": 1212384361,
    "lowbit-L16.pt": 1212384361,
    "lowbit-L17.pt": 1212384361,
    "lowbit-L18.pt": 1212384361,
    "lowbit-L19.pt": 1212384361,
    "lowbit-L20.pt": 1212384361,
    "lowbit-L21.pt": 1212384361,
    "lowbit-L22.pt": 1212384361,
    "lowbit-L23.pt": 1212384361,
    "lowbit-L24.pt": 1212384361,
    "lowbit-L25.pt": 1212384361,
    "lowbit-L26.pt": 1212384361,
    "lowbit-L27.pt": 1212384361,
    "lowbit-L28.pt": 1212384361,
    "lowbit-L29.pt": 1212384361,
    "lowbit-L30.pt": 1212384361,
    "lowbit-L31.pt": 1212384361,
    "lowbit-L32.pt": 1212384361,
    "lowbit-L33.pt": 1212384361,
    "lowbit-L34.pt": 1212384361,
    "lowbit-L35.pt": 1212384361,
    "lowbit-L36.pt": 1212384361,
    "lowbit-L37.pt": 1212384361,
    "lowbit-L38.pt": 1212384361,
    "lowbit-L39.pt": 1212384361,
    "lowbit-L40.pt": 1212384361,
    "lowbit-L41.pt": 1212384361,
    "lowbit-L42.pt": 1212384361,
    "lowbit-L43.pt": 1212384361,
    "lowbit-L44.pt": 1212384361,
    "lowbit-L45.pt": 1212384361,
    "merges.txt": 1671853,
    "model-00000.safetensors": 4611468232,
    "model-00001.safetensors": 4164305560,
    "model-00002.safetensors": 2464687296,
    "model.safetensors.index.json": 101543,
    "preprocessor_config.json": 626,
    "pruning.json": 31114,
    "special_tokens_map.json": 667,
    "tokenizer.json": 11434355,
    "tokenizer_config.json": 16798,
    "vocab.json": 2776833,
}

## Helpers
Input discovery, mode checks, the launcher process, stage waits, and the submission check. Only the standard
library is used: this kernel's Python just starts the runtime, which brings its own Python 3.12 environment.

In [ ]:
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def walk_inputs(root):
    """every file under root, sorted: one walk (symlinked dirs followed, hidden names skipped), the same list the
    launcher builds for its own lookups (it reuses the cached copy written here)"""
    out = []
    for r, dns, fns in os.walk(root, followlinks=True):
        dns[:] = sorted(d for d in dns if not d.startswith("."))
        out += [os.path.join(r, f) for f in fns if not f.startswith(".")]
    return sorted(out)


def checkpoint_problems(files):
    """{name: path} of one candidate checkpoint vs CKPT_FILES (every file, exact size): list of problems"""
    p = [f"missing {n}" for n in CKPT_FILES if n not in files]
    p += [f"{n}: {os.path.getsize(files[n])} bytes, expected {sz}" for n, sz in CKPT_FILES.items()
          if n in files and os.path.getsize(files[n]) != sz]
    return p


def pick_checkpoint(files):
    """(checkpoint dir, source, ignored dirs). The checkpoint is the Kaggle Model MODEL_SLUG split into the variations
    MODEL_PARTS (one version each). Their files are linked into CKPT_MERGE_DIR, one local directory that the launcher
    stages and serves from (no bytes are copied here). Every file of CKPT_FILES must be there with its exact size."""
    pat = re.compile(r"/" + re.escape(MODEL_SLUG) + r"/pytorch/([a-z0-9-]+)/(\d+)$")
    mounts = {}
    for d in sorted({os.path.dirname(p) for p in files}):
        m = pat.search(d.lower())
        if m:
            mounts.setdefault(m.group(1), []).append(d)
    missing = [v for v in MODEL_PARTS if v not in mounts]
    assert not missing, (f"checkpoint parts not attached: {missing}. Add the Kaggle Model {MODEL_OWNER}/{MODEL_SLUG} "
                         f"(framework PyTorch), variations {', '.join(MODEL_PARTS)}, to this notebook. "
                         f"Model variations seen: {sorted(mounts) or 'none'}")
    for v in MODEL_PARTS:
        assert len(mounts[v]) == 1, f"several versions of the Model variation {v} attached: {mounts[v]}"
    by_dir = {}
    for p in files:
        by_dir.setdefault(os.path.dirname(p), {})[os.path.basename(p)] = p
    merged = {}
    for v in MODEL_PARTS:
        for n, p in by_dir.get(mounts[v][0], {}).items():
            if n in CKPT_FILES:
                assert n not in merged or os.path.getsize(merged[n]) == os.path.getsize(p), f"{n} differs between parts"
                merged.setdefault(n, p)
    bad = checkpoint_problems(merged)
    assert not bad, "the Model parts do not form the checkpoint: " + "; ".join(bad[:8])
    if CKPT_MERGE_DIR.exists():
        for f in CKPT_MERGE_DIR.iterdir():
            f.unlink()
    CKPT_MERGE_DIR.mkdir(parents=True, exist_ok=True)
    for n, p in merged.items():
        os.symlink(p, CKPT_MERGE_DIR / n)
    used = {mounts[v][0] for v in MODEL_PARTS}
    other = sorted({os.path.dirname(p) for p in files if os.path.basename(p) == CKPT_MARKER} - used)
    return str(CKPT_MERGE_DIR), f"Kaggle Models, {len(MODEL_PARTS)} parts linked into one directory", other


def mode_problems(m):
    """differences between the runtime's mode file and this notebook's configuration (empty = identical)"""
    e = m.get("env", {})
    p = []
    checks = [("mode sha256", hashlib.sha256(json.dumps({k: v for k, v in m.items() if k != "note"}, sort_keys=True)
                                              .encode()).hexdigest(), MODE_SHA256),
              ("solve_args", m.get("solve_args"), SOLVER_ARGS), ("serve args", m.get("args"), SERVE_ARGS),
              ("deadline_total_s", m.get("deadline_total_s"), DEADLINE_TOTAL_S),
              ("startup_guard", m.get("startup_guard"), STARTUP_GUARD), ("prefetch", m.get("prefetch"), PREFETCH),
              ("gate_pins", m.get("gate_pins"), GATE_PINS), ("host_ar_gate", m.get("host_ar_gate"), HOST_ALLREDUCE_GATE),
              ("kv_cache_gib", m.get("kv_cache_gib"), KV_CACHE_GIB), ("util", m.get("util"), GPU_MEMORY_UTILIZATION),
              ("delta_file", m.get("delta_file"), DELTA_FILE), ("triton_cache_tar", m.get("triton_cache_tar"), TRITON_CACHE),
              ("ready_poll_s", m.get("ready_poll_s"), READY_POLL_S), ("ckpt_marker", m.get("ckpt_marker"), CKPT_MARKER),
              ("env keys", sorted(e), sorted(ENV_FLAGS)),
              ("W8A8 tile table", hashlib.sha256(json.dumps(e.get("ARC_W8A8_V2_CONFIG"), sort_keys=True).encode()).hexdigest(),
               W8A8_TILE_TABLE_SHA256),
              ("lowbit tile table", hashlib.sha256(json.dumps(m.get("lowbit_tune_table"), sort_keys=True).encode()).hexdigest(),
               LOWBIT_TILE_TABLE_SHA256)]
    checks += [(f"env {k}", e.get(k), v) for k, v in ENV_FLAGS.items() if k != "ARC_W8A8_V2_CONFIG"]
    for name, got, want in checks:
        if got != want:
            p.append(f"{name}: runtime {str(got)[:80]} != notebook {str(want)[:80]}")
    return p


def submission_mode_guard(j):
    """the runtime launcher's own release check (unchanged): refuses a stale or different code bundle"""
    _e, _a = j.get("env", {}), j.get("solve_args", [])
    _args = j.get("args", [])
    _spec = [json.loads(_args[i + 1]) for i, x in enumerate(_args[:-1]) if x == "--speculative-config"]
    _comp = [json.loads(_args[i + 1]) for i, x in enumerate(_args[:-1]) if x == "--compilation-config"]
    assert (j.get("lowbit_fuse_gate") is False and j.get("gate_pins", {}).get("ARC_LOWBIT_FUSE") == "1" and j.get("kv_cache_gib") == 3.0
            and "--protect" not in _a and _e.get("ARC_SESSION_EP") == "1" and "--session-chunk" in _a and j.get("swa_cuda_cands") == "1,31"
            and _e.get("ARC_AG_U16") == "1" and _e.get("ARC_HOST_AR_TILE") == "4096"
            and j.get("prefetch") and j.get("host_ar_gate") is True and _a[_a.index("--endgame-s") + 1] == "450"
            and "--temp" not in _a and "--alt" not in _a and j.get("delta_file") == "p1s_delta.safetensors"
            and _e.get("ARC_HOST_AR_BITS") == "6" and _e.get("ARC_OFFLOAD_ROW_FMT") == "e2m2"
            and _e.get("ARC_OFFLOAD_CACHE_ROWS_L", "").startswith("0:202000,1:362600,") and _e.get("ARC_OFFLOAD_WIN_LAZY") == "1"
            and _e.get("ARC_OFFLOAD_ATTN2") == "1" and _e.get("ARC_OFFLOAD_ATTN2_KERN") == "5"
            and _e.get("ARC_SHARD_SAMPLE_NOSYNC") == "1" and _e.get("ARC_OFFLOAD_AR_GATE") == "1" and _e.get("ARC_OFFLOAD_PF_COMPACT") == "32768"
            and _e.get("ARC_W8A8_V2") == "1" and json.loads(_e.get("ARC_W8A8_V2_CONFIG", "{}"))
            and j.get("gate_pins", {}).get("ARC_FP8_W8A8") == "1"
            and len(_spec) == 1 and _spec[0].get("num_speculative_tokens") == 3
            and len(_comp) == 1 and max(_comp[0].get("cudagraph_capture_sizes", [0])) == 256
            and "--tail2" not in _a and j["prefetch"].get("stage") == "always" and j["prefetch"].get("robust") is True
            and j["prefetch"].get("stage_env") is True and j["prefetch"].get("stage_threads") == 32
            and j.get("startup_guard", {}).get("deadline_abs_s") == 42150 and j.get("deadline_total_s") == 42150 and j["startup_guard"].get("kill_by_s") == 42600
            and j["startup_guard"].get("anchor") == "session" and j.get("triton_cache_tar") == "triton_cache_lv5.tarbin"
            and _e.get("ARC_AR_LB") == "1" and _e.get("ARC_AR_LB_MAIN") == "4f/5" and _e.get("ARC_AR_LB_EF") == "1"
            and _e.get("ARC_AR_LB_HAD") == "8" and "ARC_AR_LB_MAXNREG" not in _e and "ARC_AR_LB_DRAFT" not in _e
            and j.get("ready_poll_s") == 2 and _e.get("ARC_KVB_REBUILD") == "1" and _e.get("ARC_PENALTY_LAZY") == "1"
            and _e.get("ARC_MLA_WS_TOKENS") == "16384" and _e.get("ARC_PARAM_COMPACT") == "1"
            and _e.get("ARC_OFFLOAD_CCACHE") == "1" and _e.get("ARC_OFFLOAD_PF_OUT") == "1"
            and _a[_a.index("--budget-margin-s") + 1] == "300"), "stale runtime: mode_submit.json is not the release mode"
    # every per-layer row-cache size must be a multiple of the cache ways: otherwise the layer silently falls back to a
    # direct-mapped cache, which the compact (e2m2) host rows refuse at engine start
    _w = int(_e.get("ARC_OFFLOAD_WAYS", "1"))
    _rows = [int(x.split(":")[1]) for x in _e.get("ARC_OFFLOAD_CACHE_ROWS_L", "").split(",") if x] + [int(_e.get("ARC_OFFLOAD_CACHE_ROWS", "0"))]
    assert _w > 1 and all(r % _w == 0 for r in _rows), f"runtime mode: row-cache sizes {sorted(set(_rows))} not all multiples of ARC_OFFLOAD_WAYS={_w}"


def proc_start(pid):
    try:
        st = open(f"/proc/{pid}/stat").read().rpartition(")")[2].split()
        btime = next(int(l.split()[1]) for l in open("/proc/stat") if l.startswith("btime"))
        return btime + int(st[19]) / os.sysconf("SC_CLK_TCK")
    except Exception:
        return None


def session_start():
    """S0 as the launcher computes it (anchor "session"): the earliest of now, this process's start and the container
    init's start, if within anchor_max_age_s"""
    now = time.time()
    s0 = now
    for pid in (os.getpid(), 1):
        t = proc_start(pid)
        if t is not None and now - float(STARTUP_GUARD.get("anchor_max_age_s", 3600)) <= t <= now + 2:
            s0 = min(s0, t)
    return s0

In [ ]:
# variables the Jupyter kernel adds to its own environment; the launcher gets the environment a plain script would
KERNEL_ONLY_ENV = ("JPY_", "PYDEVD_", "IPY_")
KERNEL_SET_ENV = {"MPLBACKEND": "module://matplotlib_inline", "TERM": "xterm-color", "CLICOLOR": "1", "PAGER": "cat",
                  "GIT_PAGER": "cat", "FORCE_COLOR": None}


def launcher_env(extra):
    env, dropped = dict(os.environ), []
    for k in list(env):
        v = KERNEL_SET_ENV.get(k, "")
        if k.startswith(KERNEL_ONLY_ENV) or (k in KERNEL_SET_ENV and (v is None or env[k].startswith(v))):
            env.pop(k); dropped.append(k)
    env.update(extra)
    return env, sorted(dropped)


def descendants(root):
    kids = {}
    for d in os.listdir("/proc"):
        if d.isdigit():
            try:
                pp = int(open(f"/proc/{d}/stat").read().rpartition(")")[2].split()[1])
                kids.setdefault(pp, []).append(int(d))
            except Exception:
                pass
    out, todo = [], [root]
    while todo:
        for c in kids.get(todo.pop(), []):
            out.append(c); todo.append(c)
    return out


class Runner:
    """the runtime launcher (run_l4.py) as a child process, plus its log file"""

    def __init__(self, proc, run_log, offset, s0, kill_by):
        self.proc, self.run_log, self.offset, self.s0, self.kill_by = proc, run_log, offset, s0, kill_by
        self.seen, self.backstop = "", False

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def read_log(self):
        try:
            with open(self.run_log, errors="replace") as f:
                f.seek(self.offset)
                self.seen += f.read()
                self.offset = f.tell()
        except OSError:
            pass
        return self.seen

    def poll(self):
        """backstop: the launcher ends itself at S0 + kill_by_s; if it is still alive BACKSTOP_GRACE_S later, kill it
        (and everything under it) here and repair the submission"""
        if self.alive() and self.kill_by is not None and time.time() > self.kill_by + BACKSTOP_GRACE_S and not self.backstop:
            self.backstop = True
            pids = descendants(self.proc.pid)
            for p in pids + [self.proc.pid]:
                try:
                    os.kill(p, 9)
                except OSError:
                    pass
            log(f"BACKSTOP: launcher alive {BACKSTOP_GRACE_S} s after its hard stop, killed {len(pids) + 1} processes")
            self.proc.wait()


STAGES = {   # launcher log lines that end each stage (any later stage, or the launcher's exit, ends it too)
    "environment": [r"re-exec under "],
    "install": [r"startup guard: ", r"triton cache restore"],
    "server start": [r"server pid \d+"],
    "server ready": [r"server ready after "],
    "solver": [r"solver (RERUN|save-run) "],
    "done": [r"\] done$", r"kill-by reached", r"-> no restart", r"solver skipped"],
}
ORDER = list(STAGES)


def reached(text, stage):
    i = ORDER.index(stage)
    return any(re.search(p, text, re.M) for s in ORDER[i:] for p in STAGES[s])


def wait_for(runner, stage, status_every_s=None):
    """block until the launcher has passed `stage` (or exited); returns the elapsed seconds"""
    t0 = time.time()
    last = t0
    while runner.proc is not None:
        runner.poll()
        if reached(runner.read_log(), stage) or not runner.alive():
            break
        if status_every_s and time.time() - last >= status_every_s:
            last = time.time()
            log(progress(runner))
        time.sleep(5)
    if runner.proc is not None and not runner.alive():
        runner.read_log()
    return time.time() - t0


def tail_json_lines(path, n=2, nbytes=8192):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2); f.seek(max(0, f.tell() - nbytes))
            return [json.loads(l) for l in f.read().decode(errors="replace").splitlines()[-n:] if l.startswith("{")]
    except Exception:
        return []


def progress(runner):
    m = tail_json_lines(WORK_DIR / "metrics.jsonl")
    s = f"S0 + {time.time() - runner.s0:.0f} s"
    if len(m) == 2 and m[1]["t"] > m[0]["t"]:
        s += f" | decode {(m[1]['gen'] - m[0]['gen']) / (m[1]['t'] - m[0]['t']):.0f} tok/s, running {m[1]['running']:.0f}"
    try:
        s += f" | attempts finished {sum(1 for l in open(WORK_DIR / 'solve_log.txt') if '] done ' in l)}"
    except OSError:
        pass
    return s


def stage_line(runner, stage, pats):
    """the launcher's own log line for a stage, for the stage cell's summary"""
    for p in pats:
        hit = re.findall(r"^.*" + p + r".*$", runner.seen, re.M)
        if hit:
            return hit[-1][:200]
    rc = "not launched" if runner.proc is None else "running" if runner.alive() else f"launcher exit rc {runner.proc.returncode}"
    return f"({stage}: no log line, {rc})"


def wait_exit(runner):
    """block until the launcher has exited (it ends itself at its hard stop; the backstop covers the rest)"""
    while runner.proc is not None and runner.alive():
        runner.poll()
        time.sleep(5)
    runner.read_log()


def monitor(fn, *a, **k):
    """stage cells only observe the launcher: an error here is logged and never stops the notebook (a failed cell
    would end the kernel, and the launcher with it, before the submission check)"""
    try:
        return fn(*a, **k)
    except Exception as ex:
        log("notebook monitor error (the launcher is not affected):", repr(ex)[:300])


def is_grid(g):
    return isinstance(g, list) and g and all(isinstance(r, list) and r and all(isinstance(v, int) for v in r) for r in g)


def check_submission(challenges_file, path, repair=True):
    """same rule as the launcher's own check: every test input of the challenges file with two grids; a missing or
    malformed attempt becomes the input echo (valid ones are kept); rewritten atomically only if something changed"""
    ch = json.load(open(challenges_file))
    try:
        sub = json.load(open(path))
        sub = sub if isinstance(sub, dict) else {}
    except Exception:
        sub = {}
    out, fixed, echo, n = {}, 0, 0, 0
    for t, v in ch.items():
        cur = sub.get(t) if isinstance(sub.get(t), list) else []
        outs = []
        for k, x in enumerate(v["test"]):
            e = cur[k] if k < len(cur) and isinstance(cur[k], dict) else {}
            a1 = e.get("attempt_1") if is_grid(e.get("attempt_1")) else None
            a2 = e.get("attempt_2") if is_grid(e.get("attempt_2")) else None
            fixed += (a1 is None) + (a2 is None)
            outs.append({"attempt_1": a1 if a1 is not None else x["input"], "attempt_2": a2 if a2 is not None else x["input"]})
            echo += (outs[-1]["attempt_1"] == x["input"]) + (outs[-1]["attempt_2"] == x["input"]); n += 2
        out[t] = outs
    if repair and (fixed or set(sub) != set(out)):
        json.dump(out, open(str(path) + ".tmp", "w"))
        os.replace(str(path) + ".tmp", path)
    return {"tasks": len(out), "test inputs": n // 2, "attempts repaired": fixed, "attempts equal to the input": echo}

## Stage 1: inputs
Find the runtime bundle (code and Python environment), the competition data and the checkpoint. The checkpoint is the
Kaggle Model *dots3-note-lb144r*, split into 4 variations (`lb144r-p1` .. `lb144r-p4`, about 15 GB each). Their files
are linked into one local directory: links only, no bytes are copied here. The launcher stages and serves from that
directory. Every one of the 63 checkpoint files must be present with its exact size, or the notebook stops here with a
message naming what is missing. Several runtime bundles, environments or versions of a part are also an error.

In [ ]:
def discover_inputs():
    files = walk_inputs(INPUT_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    json.dump({"in": INPUT_DIR, "files": files}, open(WORK_DIR / "input_index.json", "w"))   # the launcher's lookup cache
    code = sorted({os.path.dirname(p) for p in files if os.path.basename(p) == "run_l4.py"})
    assert len(code) == 1, f"expected exactly one runtime bundle (dataset {RUNTIME_DATASET}), found {code}"
    env = [p for p in files if re.search(r"/arc_py312\.tar[^/]*$", p)]
    assert len(env) == 1, f"expected exactly one Python environment archive, found {env}"
    ch = [p for p in files if p.endswith("/arc-agi_test_challenges.json")]
    assert ch, "competition data not attached"
    ckpt, src, other = pick_checkpoint(files)
    mode = json.load(open(Path(code[0]) / MODE_FILE))
    submission_mode_guard(mode)
    bad = mode_problems(mode)
    assert not bad, "runtime mode differs from this notebook's configuration:\n  " + "\n  ".join(bad)
    inp = {"code": code[0], "env_archive": env[0], "challenges": ch[0], "checkpoint": ckpt, "checkpoint_source": src,
           "ignored_checkpoints": other, "n_files": len(files), "mode": mode}
    gib = sum(CKPT_FILES.values()) / 2**30
    for k in ("code", "env_archive", "challenges", "checkpoint", "checkpoint_source", "ignored_checkpoints"):
        log(f"{k:20s}", inp[k])
    log(f"{'checkpoint size':20s} {gib:.1f} GiB, all {len(CKPT_FILES)} files present with their exact sizes; mode identical to the configuration above ({len(ENV_FLAGS)} env flags)")
    return inp


t_nb = time.time()
INPUTS = discover_inputs()

## Stage 2: run mode
- **Scored rerun** (`KAGGLE_IS_COMPETITION_RERUN` set): the hidden test set with the runtime's submission mode as is.
- **Save run**: the same stack on the public evaluation split with a 1200 s solver deadline (a smoke test of start-up,
  memory and throughput). It is stopped after 1800 s at most, and `submission.json` stays the input echo.

In [ ]:
def resolve_run_mode(inp):
    rerun = bool(os.environ.get("KAGGLE_IS_COMPETITION_RERUN"))
    if rerun:
        mode_file, mode = MODE_FILE, dict(inp["mode"])          # read by the launcher relative to the runtime bundle
    else:
        mode = dict(inp["mode"]); mode.pop("save_overrides", None)
        mode.update(SAVE_RUN)
        mode_file = str(WORK_DIR / "mode_diag.json")
        json.dump(mode, open(mode_file, "w"))
    s0 = session_start()
    sg = mode["startup_guard"]
    run = {"rerun": rerun, "mode_file": mode_file, "challenges": mode["challenges"], "s0": s0,
           "solver_deadline_s0": min(time.time() - s0 + mode["deadline_total_s"], sg["deadline_abs_s"]),
           "kill_by_s0": sg["kill_by_s"]}
    log("run:", "SCORED RERUN (hidden test set)" if rerun else "SAVE RUN (public evaluation split, short deadline)")
    log(f"session start S0 = now - {time.time() - s0:.0f} s; solver deadline <= S0 + {run['solver_deadline_s0']:.0f} s; "
        f"hard stop S0 + {run['kill_by_s0']} s")
    log("solver args:", " ".join(mode["solve_args"]))
    return run


RUN = resolve_run_mode(INPUTS)

## Stage 3: runtime launch and environment
Start the launcher as a child process. Before handing over to the bundled Python 3.12 environment, it:
1. starts the parallel copier: the env, CUDA and Triton tars first, then the delta, the safetensors and the 45 expert
   files, all to local disk;
2. unpacks the environment;
3. re-executes itself under that environment, then installs the vLLM patches, the plugin and the CUDA toolkit, and
   restores the Triton cache.

The launcher must be a child process: it replaces its own process image (`exec`) and ends with a hard `_exit` at the
deadline, and neither may happen inside this notebook's kernel. Its output streams into this cell's log, and its
own log is `run_log.txt` in the working directory.

In [ ]:
def launch_runtime(inp, run):
    run_log = WORK_DIR / "run_log.txt"
    offset = run_log.stat().st_size if run_log.exists() else 0
    env, dropped = launcher_env({"ARC_MODE_FILE": run["mode_file"], "ARC_CKPT_DIR": inp["checkpoint"]})
    cmd = [sys.executable, os.path.join(inp["code"], "run_l4.py")]
    plan = {"cmd": cmd, "ARC_MODE_FILE": run["mode_file"], "ARC_CKPT_DIR": inp["checkpoint"], "env_dropped": dropped,
            "rerun": run["rerun"], "save_watchdog": None if run["rerun"] else [SAVE_WATCHDOG_S, SAVE_WATCHDOG_KILL_S]}
    json.dump(plan, open(WORK_DIR / "notebook_plan.json", "w"), indent=1)
    log("launch:", " ".join(cmd), "| ARC_MODE_FILE =", run["mode_file"], "| kernel-only env removed:", dropped or "none")
    if DRY_RUN:
        log("DRY RUN: not launched")
        return Runner(None, run_log, offset, run["s0"], run["s0"] + run["kill_by_s0"])
    proc = subprocess.Popen(cmd, env=env)
    if not run["rerun"]:   # save run: bounded wall time whatever happens inside
        subprocess.Popen(["sh", "-c", f"sleep {SAVE_WATCHDOG_S}; kill -TERM {proc.pid} 2>/dev/null; "
                                      f"sleep {SAVE_WATCHDOG_KILL_S}; kill -9 {proc.pid} 2>/dev/null"],
                         start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return Runner(proc, run_log, offset, run["s0"], run["s0"] + run["kill_by_s0"])


RUNNER = launch_runtime(INPUTS, RUN)
monitor(wait_for, RUNNER, "install")
monitor(lambda: log("environment:", stage_line(RUNNER, "environment", STAGES["environment"])))
monitor(lambda: log("install:    ", stage_line(RUNNER, "install", [r"triton cache restored", r"startup guard: "])))

## Stage 4: model server
The launcher measures the one remaining startup gate (host vs NCCL all-reduce) and then starts `vllm serve` on the
local checkpoint mirror. The ranks wait for each expert file while the copier is still writing it. **Startup guard:**
if the server dies or hangs before it is ready, it is restarted once (only if at least 1800 s are left). If it cannot
be ready with at least 120 s left to solve, the launcher keeps the fallback submission and exits.

In [ ]:
def server_stage(runner):
    wait_for(runner, "server ready")
    log("server:", stage_line(runner, "server ready", [r"server ready after ", r"-> no restart", r"server died"]))
    for line in re.findall(r"^.*-> restart \d.*$", runner.seen, re.M):
        log("startup guard:", line[:200])


monitor(server_stage, RUNNER)

## Storage diagnostics
Where each input lives, and how fast it was read. For each checkpoint part and the runtime dataset, this cell shows
the underlying mount (filesystem type, source, mount point) and the read speed during the local staging copy. The
speed is computed from the copier's per-file log, so the checkpoint is not read again. Information only: this cell
can never fail the run.

In [ ]:
def _sanitise(src):
    """mount sources may carry tokens or credentials: keep the shape, drop the secrets"""
    s = re.sub(r"(?i)(token|key|secret|password|passwd|auth|sig|signature|credential)[^,;&\s]*=[^,;&\s]*", r"\1=<redacted>", src)
    s = re.sub(r"//[^/@\s]+@", "//<redacted>@", s)
    s = re.sub(r"[A-Za-z0-9+/_=-]{32,}", "<redacted>", s)
    return s[:120]


def mount_of(path):
    """(fstype, source, mount point) of the longest-prefix mount holding realpath(path), from /proc/self/mountinfo"""
    p, best = os.path.realpath(path), None
    for line in open("/proc/self/mountinfo"):
        pre, _, post = line.partition(" - ")
        mp = pre.split()[4].replace("\\040", " ")
        if (p == mp or p.startswith(mp.rstrip("/") + "/")) and (best is None or len(mp) >= len(best[2])):
            f = post.split()
            best = (f[0] if f else "?", _sanitise(f[1]) if len(f) > 1 else "?", mp)
    return best or ("?", "?", "?")


def storage_report(inp):
    """per source: mount + read speed of the staging copy (from prefetch_log.txt 'staged' lines; the interval up to
    each file's completion is attributed to that file's source, at the aggregate rate the copier achieved in it)"""
    src_of, label = {}, {}
    md = Path(inp["checkpoint"])
    for f in md.iterdir():
        if f.is_symlink():
            part_dir = os.path.dirname(os.readlink(f))
            m = re.search(r"/pytorch/([^/]+)/\d+$", part_dir.lower())
            src_of[f.name] = m.group(1) if m else part_dir
            label[src_of[f.name]] = part_dir
    label["runtime"] = inp["code"]
    for n in os.listdir(inp["code"]):
        src_of.setdefault(n, "runtime")
    rows = {s: {"dir": d, "bytes": 0, "seconds": 0.0, "files": 0} for s, d in label.items()}
    lines = (WORK_DIR / "prefetch_log.txt").read_text().splitlines() if (WORK_DIR / "prefetch_log.txt").exists() else []
    t_prev, cum_prev, total, t_end = 0.0, 0.0, 0.0, 0.0
    for l in lines:
        m = re.match(r"\[\s*([\d.]+)s\] staged (\S+) ([\d.]+) GiB cum ([\d.]+) GiB", l)
        if not m:
            continue
        t, name, size, cum = float(m.group(1)), m.group(2), float(m.group(3)) * 2**30, float(m.group(4)) * 2**30
        r = rows.setdefault(src_of.get(name, "other"), {"dir": "", "bytes": 0, "seconds": 0.0, "files": 0})
        r["bytes"] += cum - cum_prev; r["seconds"] += t - t_prev; r["files"] += 1
        t_prev, cum_prev, total, t_end = t, cum, cum, t
    out = []
    for s, r in rows.items():
        try:
            fs, msrc, mp = mount_of(r["dir"]) if r["dir"] else ("?", "?", "?")
        except Exception as ex:   # no mount table: keep the speeds
            fs, msrc, mp = "?", f"unavailable ({type(ex).__name__})", "?"
        mbps = r["bytes"] / 2**20 / r["seconds"] if r["seconds"] > 0 else None
        out.append({"source": s, "fstype": fs, "mount_source": msrc, "mount_point": mp, "staged_files": r["files"],
                    "staged_gib": round(r["bytes"] / 2**30, 2), "read_mb_s": None if mbps is None else round(mbps)})
    overall = {"staged_gib": round(total / 2**30, 2), "seconds": round(t_end, 1),
               "read_mb_s": round(total / 2**20 / t_end) if t_end > 0 else None}
    json.dump({"sources": out, "overall": overall}, open(WORK_DIR / "storage_diagnostics.json", "w"), indent=1)
    log(f"{'source':10s} {'fstype':16s} {'files':>5s} {'GiB':>6s} {'MB/s':>6s}  mount point | source")
    for o in out:
        log(f"{o['source']:10s} {o['fstype']:16s} {o['staged_files']:5d} {o['staged_gib']:6.2f} "
            f"{str(o['read_mb_s']):>6s}  {o['mount_point']} | {o['mount_source']}")
    log(f"{'overall':10s} {'':16s} {'':5s} {overall['staged_gib']:6.2f} {str(overall['read_mb_s']):>6s}  "
        f"({overall['seconds']} s of staging; per-source speeds are the copier's aggregate rate while that source's "
        f"files completed)")
    return out


try:
    STORAGE = monitor(storage_report, INPUTS)
except BaseException as ex:   # never stop the notebook here
    log("storage diagnostics unavailable:", repr(ex)[:200])

## Stage 5: solver
`solve.py` runs until its deadline (S0 + 42,150 s at most in the scored run) and writes `submission.json`. A status
line appears here every 15 minutes. At S0 + 42,600 s the launcher stops everything, repairs the submission and exits.

In [ ]:
monitor(wait_for, RUNNER, "done", status_every_s=None if DRY_RUN else 900)
wait_exit(RUNNER)
monitor(lambda: log("solver:", stage_line(RUNNER, "solver", [r"solver exit rc.*", r"solver skipped.*", r"kill-by reached.*"])))
log("launcher exit code:", None if RUNNER.proc is None else RUNNER.proc.returncode)

## Stage 6: submission check
Check `submission.json` against the test challenges: every test input needs two grids. This is the same rule the
launcher applies; anything missing or malformed becomes the input echo, so the file always scores.

In [ ]:
def report_run(run):
    """save-run score line, decode speed and memory from the launcher's logs (information only)"""
    if not run["rerun"]:
        try:
            log("save-run smoke:", [l.strip() for l in open(WORK_DIR / "solve_log.txt") if "local score" in l][-1])
        except (OSError, IndexError):
            log("save-run smoke: no local score line")
    m = [json.loads(l) for l in open(WORK_DIR / "metrics.jsonl")] if (WORK_DIR / "metrics.jsonl").exists() else []
    m = [x for x in m if x.get("running", 0) > 0]
    if len(m) > 1 and m[-1]["t"] > m[0]["t"]:
        log(f"decode: {(m[-1]['gen'] - m[0]['gen']) / (m[-1]['t'] - m[0]['t']):.0f} tok/s average while requests ran, "
            f"min free GPU memory {min((x.get('gpu_free_mib_min', 1e9) for x in m), default=0):.0f} MiB")
    log(f"notebook wall time {time.time() - t_nb:.0f} s (S0 + {time.time() - run['s0']:.0f} s)")


def validate_submission(inp, run):
    path = WORK_DIR / "submission.json"
    rep = check_submission(inp["challenges"], path)
    log("submission.json:", json.dumps(rep), f"{path.stat().st_size / 1024:.0f} KiB")
    monitor(report_run, run)
    return rep


SUBMISSION = validate_submission(INPUTS, RUN)